---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-31: LangChain <b>Prompts</b> Component</h1>

# Learning agenda of this notebook  

1. Understanding LangChain Prompts Component
    - Dynamic Reusable Prompts (`langchain.prompts.PromptTemplate`)
        - Example 1: Dynamic Prompts using **f-strings**
        - Example 2: Dynamic Prompts with LangChain's `PromptTemplate`
        - Example 3: Saving LangChain PromptTemplate and loading and reusing them dynamically
        - Example 4: A Research Paper Summarizer that uses LangChain `PromptTemplate` with gradio
    - Role-Based Conversational Prompts (`langchain.prompts.ChatPromptTemplate`)
        - Example 5: Understanding LangChain messages: `SystemMessage`, `HumanMessage` and `AIMessage`
        - Example 6: A Q/A Bot **Without** Chat History
        - Example 7: A Conversational ChatBot **With** Chat History
        - Example 8: A Conversational ChatBot that uses LangChain's `ChatPromptTemplate` **With** Chat History

# <span style='background :lightgreen' >Recap: Core Components of [LangChain](https://github.com/langchain-ai/langchain)</span>


<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">LangChain is an open source framework that provides us with a set of tools and abstractions that make it easier for us to create complex LLM-powered applications.</h3>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Langchain components can be chained together to create sophisticated AI applications like chatbots, question-answering systems, and intelligent agents.</h3>


| Component | Description | Key Benefit |
|-----------|--------------|-------------|
| **Models** | LangChain's "universal remote control" for AI—one interface to rule them all, freeing developers from vendor-specific complexity and enabling true AI provider independence. | Write once, use with any AI model |
| **Prompts** | Reusable, parameterized templates that standardize how you communicate with AI models. | Consistent messaging with easy updates | 
| **Memory** | LangChain Memory enables AI models to maintain context across multiple interactions in a conversation. Without memory, each interaction is independent - the model has no knowledge of previous exchanges. | Natural, flowing conversations |
| **Chains** | LangChain’s “AI Assembly Lines” — they connect multiple models, prompts, or tools together into a logical flow of reasoning, automating complex multi-step tasks with simplicity and consistency. | Transform complex tasks into simple sequences | 
| **Indexes** | Indexes are LangChain's smart knowledge management systems that convert your raw documents (PDFs, websites, databases) into searchable libraries where AI models can quickly find and retrieve only the most relevant information w/o exceeding context limits. | Fast retrieval, reduced API costs | 
| **Agents** | LangChain Agents are the “decision-making brains” of your AI system — they enable models to reason, plan, and dynamically decide which tools to use and what actions to take in order to achieve a user’s goal.| Autonomous problem-solving without manual programming |

# <span style='background :lightgreen' >1. Overview of LangChain **Prompts** Component</span>

<h3 align="center"><div class="alert alert-success" style="margin: 20px">Prompts are input instructions or queries given to a model to guide its output</h3>

<h3 align="center"><div class="alert alert-success" style="margin: 20px">Prompt templates allow you to create reusable prompts with dynamic placeholders that get filled in at runtime. This makes prompts flexible, testable, and easier to iterate on</h3>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">LangChain Prompts are <b>reusable, parameterized templates</b> that standardize how you communicate with AI models</h3>

- Prompt templates act like “fill-in-the-blank” patterns where you plug variables into a fixed structure instead of writing a new prompt from scratch every time.
- Instead of manually writing full prompts each time, you reuse this structure with different values for the placeholders.
- You can format your prompt template with input variables using either of the following two options:
    - Use **Python f-strings** for simple prompts with basic variable substitution. By prefixing a string with f or F, you can embed expressions within curly braces ({}), which are evaluated at runtime.
    - Use **mustache format** (https://mustache.github.io/mustache.5.html) that provides features for handling complex data structures and logic, which is helpful for evaluators and advanced use cases.
- Examples of some simple prompt templates might look like this:
```python
"As a {role}, explain {topic} in {style} for {audience}."
"Here is some context:\n{context}\nAnswer this question:\n{question}."
"You are a customer support agent. This is the refund policy: {refund_policy}. Please respond to the user's question: {question}"
"Analyze {input} and return JSON with keys: {keys_list}."
```
- **Prompt templates are widely used in:**
    - Public prompt repositories, where reusable and interactive prompt templates are shared by the community:
        - https://prompts.chat/
        - https://github.com/f/prompts.chat
        - https://www.promptbase.com/
    - MCP (Model Context Protocol) servers, which expose reusable and parameterized prompt templates via MCP protocol with well-defined input/output schemas, enabling dynamic prompt discovery and sharing across AI applications.
    - Frameworks like LangChain, which provide abstractions such as `PromptTemplate` and `ChatPromptTemplate` to programmatically construct, reuse, and compose prompts.
    - LangGraph workflows, where prompt templates (including MCP-based prompts) are integrated as message templates within stateful, multi-agent and graph-based agent architectures.

## Example 1: Dynamic Prompts using **f-strings**

In [1]:
import os                                           # Access environment variables (e.g. os.getenv("API_KEY"))
from dotenv import load_dotenv                      # Load API keys from a .env file into environment variables         
from langchain_openai import ChatOpenAI             # LangChain wrapper for OpenAI models (e.g. GPT-4o)      

load_dotenv('../keys/.env', override=True)          # Load variables from .env file into system environment
groq_api_key = os.getenv('GROQ_API_KEY')            # Retrieve Groq API key (used to access LLaMA model)
# Initialize the Chat Model (Groq’s LLaMA via OpenAI-compatible API)
llama_model = ChatOpenAI(
                        model="llama-3.3-70b-versatile",               #"llama-3.1-8b-instant"
                        api_key=groq_api_key,                           
                        base_url="https://api.groq.com/openai/v1",      # Groq provides OpenAI-compatible API endpoint
                        temperature=0.7, 
                        max_completion_tokens=256,
                        streaming=False   
                        )

# Get User Input
user_name = input("Enter the name of the person you want to greet: ")
# Create Prompt using Python f-string
prompt = f"Greet this person in five languages, one of which should be Urdu. The name of the person is {user_name}"

# Send Prompt to Model and Get Response
result = llama_model.invoke(prompt)                
print(result.content)                              

Enter the name of the person you want to greet:  Saleem Raza


Here are greetings in five languages, including Urdu:

1. English: Hello Saleem Raza, how are you?
2. Spanish: Hola Saleem Raza, ¿cómo estás?
3. French: Bonjour Saleem Raza, comment allez-vous?
4. Urdu: السلام علیکم سلیم رضا، آپ کیسے ہو؟ (Assalamu alaikum Saleem Raza, aap kaise ho?)
5. German: Hallo Saleem Raza, wie geht es Ihnen?

Note: In Urdu, "Assalamu alaikum" is a common greeting that roughly translates to "peace be upon you." It's a more formal greeting, often used as a sign of respect.


##  LangChain PromptTemplate and ChatPromptTemplate Classes

```mermaid
graph TD
    A["🧠 <b>Model</b><br/>invoke()"]
    
    %% Branch 1 - Single message
    B1["📩 Single Message<br/>(Single-turn stand-alone query)"]
    C1["🗒️ Static Message"]
    D1["⚙️ Dynamic Message<br/>(<b>PromptTemplate</b>)"]
    
    %% Branch 2 - List of messages
    B2["💬 List of Messages<br/>(Multi-turn conversation)"]
    C2["🗒️ Static Messages<br/>(System, Human, AI Messages)"]
    D2["⚙️ Dynamic Messages<br/>(<b>ChatPromptTemplate</b>)"]

    %% Connections
    A --> B1
    A --> B2
    B1 --> C1
    B1 --> D1
    B2 --> C2
    B2 --> D2

    %% Styling
    style A fill:#fff3e0,stroke:#f57c00,stroke-width:2px
    style B1 fill:#e1f5fe,stroke:#0288d1,stroke-width:2px
    style B2 fill:#ede7f6,stroke:#673ab7,stroke-width:2px
    style C1 fill:#bbdefb,stroke:#0288d1,stroke-width:1.5px
    style D1 fill:#81d4fa,stroke:#0288d1,stroke-width:1.5px
    style C2 fill:#d1c4e9,stroke:#673ab7,stroke-width:1.5px
    style D2 fill:#b39ddb,stroke:#673ab7,stroke-width:1.5px

```

In [2]:
from langchain_core.prompts import PromptTemplate                          # The standard template class used to generate dynamic text prompts by inserting variables into a template
from langchain_core.prompts import ChatPromptTemplate                      # Used to structure prompts as role-based chat conversations for chat models
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage # Used to represent system instructions, user input, and AI responses inside a chat prompt

# <span style='background :lightgreen' >2. Dynamic Reusable Prompts with LangChain's `PromptTemplate`</span>
<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">The PromptTemplate class in LangChain is a structured way to create prompts dynamically by inserting variables into a predefined template. Instead of hardcoding prompts, PromptTemplate allows you to define placeholders that can be filled in at runtime with different inputs.</h3>

- **Why to use PromptTemplate class, when this can be done using f-strings?**
    - **Ensures validation,** i.e., that all placeholders (e.g. {topic}, {audience_level}, {style}) are correctly provided at runtime — raises clear errors if missing. (F-strings fail silently or produce malformed prompts)
    - **Are reusable,** i.e., once a PromptTemplate is defined, it can be saved somewhere as a JSON file and reused later, making it ideal for pipelines or repeated tasks. 
    - **Enables integration,** with LangChain features like prompt serialization, formatting tools, few-shot templates, and structured prompt management.
- **Best For:**
    - Content generation (articles, summaries, translations)
    - Data transformation
    - Single-shot AI tasks
    - Non-conversational requests


```mermaid
graph LR
    %% Align to right
    classDef rightAlign text-align:right;

    %% ======== PROMPT TEMPLATE INVOCATION FLOW ========
    A["🧱 PromptTemplate<br/>(Defines text with placeholder {user_name})"]:::rightAlign
    B["👤 User Input<br/>(your_name = input('Enter name'))"]:::rightAlign
    C["🧩 Template Filling<br/>{user_name} → actual name"]:::rightAlign
    D["📜 Prepared Prompt<br/>(template1.invoke({'user_name': your_name}))"]:::rightAlign
    E["🤖 Chat Model<br/>ChatOpenAI(model='llama-3.1-8b-instant')"]:::rightAlign
    F["⚙️ LLM Invocation<br/>llama_model.invoke(prompt)"]:::rightAlign
    G["💬 Model Response<br/>print(result.content)"]:::rightAlign

    %% FLOW
    A -->|Waits for variable| B
    B -->|Provides value| C
    C -->|Creates| D
    D -->|Sent to| E
    E -->|Runs| F
    F -->|Returns| G

    %% STYLING
    style A fill:#e1f5fe,stroke:#0288d1,stroke-width:2px
    style B fill:#e3f2fd,stroke:#0288d1,stroke-width:1.5px
    style C fill:#bbdefb,stroke:#0288d1,stroke-width:1.5px
    style D fill:#81d4fa,stroke:#0288d1,stroke-width:1.5px
    style E fill:#4fc3f7,stroke:#0288d1,stroke-width:2px
    style F fill:#29b6f6,stroke:#0288d1,stroke-width:2px
    style G fill:#03a9f4,stroke:#0288d1,stroke-width:2px
```

## Example 2: Dynamic Prompts using LangChain's `PromptTemplate`

In [3]:
# Two ways to create a PromptTemplate in LangChain
from langchain_core.prompts import PromptTemplate   # The standard template class used to generate dynamic text prompts by inserting variables into a template

##########           Method 1: Using the constructor (manual approach)       ##############
#  The constructor requires you to explicitly specify the variables used in the template
prompt_template1 = PromptTemplate(
                                  template="Tell me a joke about {topic} for {audience}",
                                  input_variables=['topic', 'audience']
                                )


##########           Method 2: Using from_template()  (automatic approach)       ##############
# The .from_template() method automatically detects the variables from the template
prompt_template2 = PromptTemplate.from_template("Tell me a joke about {topic} for {audience}")


# Display the automatically created prompt template
print(prompt_template1)
print(prompt_template2)

input_variables=['audience', 'topic'] input_types={} partial_variables={} template='Tell me a joke about {topic} for {audience}'
input_variables=['audience', 'topic'] input_types={} partial_variables={} template='Tell me a joke about {topic} for {audience}'


In [4]:
mytopic= "parrots"
myaudience="kids"

# Returns a plain string. It simply fills the variables in the template. Output is raw text
prompt1 = prompt_template2.format(topic= mytopic, audience=myaudience)  
print(type(prompt1))
print(prompt1)

<class 'str'>
Tell me a joke about parrots for kids


In [5]:
mytopic= "parrots"
myaudience="kids"

# The 'invoke' method of PromptTemplate class substitutes {topic} with the provided value. Returns a LangChain PromptValue object, ready to be passed directly into LLMs, Chains, Pipelines
prompt2 = prompt_template2.invoke({"topic": mytopic, "audience": myaudience}) 

print(type(prompt2))
print(prompt2)

<class 'langchain_core.prompt_values.StringPromptValue'>
text='Tell me a joke about parrots for kids'


In [6]:
import os                                           # Access environment variables (e.g. os.getenv("API_KEY"))
from dotenv import load_dotenv                      # Load API keys from a .env file into environment variables         
from langchain_openai import ChatOpenAI             # LangChain wrapper for OpenAI models (e.g. GPT-4o)      
from langchain_core.prompts import PromptTemplate   # The standard template class used to generate dynamic text prompts by inserting variables into a template

load_dotenv('../keys/.env', override=True)          # Load variables from .env file into system environment
groq_api_key = os.getenv('GROQ_API_KEY')            # Retrieve Groq API key (used to access LLaMA model)
llama_model = ChatOpenAI(model="llama-3.3-70b-versatile", api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")


# At this point, 'prompt2' contains the full prompt text ready to be sent to the LLM
response = llama_model.invoke(prompt2)                 
print(response.content)  

Why did the parrot go to the doctor?

Because it had a fowl cough.


## Example 3: Dynamic Prompts using LangChain's `PromptTemplate`

In [7]:
import os                                           # Access environment variables (e.g. os.getenv("API_KEY"))
from dotenv import load_dotenv                      # Load API keys from a .env file into environment variables         
from langchain_openai import ChatOpenAI             # LangChain wrapper for OpenAI models (e.g. GPT-4o)      
from langchain_core.prompts import PromptTemplate   # The standard template class used to generate dynamic text prompts by inserting variables into a template

load_dotenv('../keys/.env', override=True)          # Load variables from .env file into system environment
groq_api_key = os.getenv('GROQ_API_KEY')            # Retrieve Groq API key (used to access LLaMA model)
llama_model = ChatOpenAI(model="llama-3.3-70b-versatile", api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")


greet_template = PromptTemplate.from_template('Greet this person in five languages, one of which should be Urdu. The name of the person is {user_name}')
your_name = input("Enter the name of the person you want to greet: ")  
prompt = greet_template.invoke({'user_name': your_name}) 

response = llama_model.invoke(prompt)                 
print(response.content)  

Enter the name of the person you want to greet:  Arif Butt


Here are greetings in five languages for Arif Butt:

1. English: Hello Arif Butt, how are you?
2. Spanish: Hola Arif Butt, ¿cómo estás?
3. French: Bonjour Arif Butt, comment allez-vous?
4. Urdu: السلام عليكمعارف بٹ، آپ کیسے ہو؟ (Assalam o Alaikum Arif Butt, aap kaise ho?)
5. German: Hallo Arif Butt, wie geht es Ihnen?


## Example 4: Dynamic Prompts using LangChain's `PromptTemplate`

In [8]:
import os                                           # Access environment variables (e.g. os.getenv("API_KEY"))
from dotenv import load_dotenv                      # Load API keys from a .env file into environment variables         
from langchain_openai import ChatOpenAI             # LangChain wrapper for OpenAI models (e.g. GPT-4o)      
from langchain_core.prompts import PromptTemplate   # The standard template class used to generate dynamic text prompts by inserting variables into a template

load_dotenv('../keys/.env', override=True)          # Load variables from .env file into system environment
groq_api_key = os.getenv('GROQ_API_KEY')            # Retrieve Groq API key (used to access LLaMA model)
llama_model = ChatOpenAI(model="llama-3.3-70b-versatile", api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")


sentiment_template = PromptTemplate.from_template("You are a sentiment analysis expert. Analyze the following sentence and classify its sentiment as Positive, Negative, or Neutral.\n Sentence:{user_sentence}")
user_sentence = input("Enter a sentence to analyze its sentiment: ")
prompt = sentiment_template.invoke({"user_sentence": user_sentence})

# Send prompt to model and print response
response = llama_model.invoke(prompt)
print(response.content)

Enter a sentence to analyze its sentiment:  Arif's YouTube videos are very informative


After analyzing the sentence, I would classify its sentiment as: **Positive**

The word "informative" has a positive connotation, implying that Arif's YouTube videos are useful, helpful, and provide valuable information. The sentence expresses a favorable opinion about Arif's content, indicating a positive sentiment.


## Example 5: Saving LangChain PromptTemplate and loading and reusing them dynamically
- The `.save()` method converts a PromptTemplate object into a JSON configuration file so that it can be:
    -  Reused and reloaded later without redefining it in code
    -  Shared with others
    -  Used in pipelines / production systems

In [5]:
import os                                           # Access environment variables (e.g. os.getenv("API_KEY"))
from dotenv import load_dotenv                      # Load API keys from a .env file into environment variables         
from langchain_openai import ChatOpenAI             # LangChain wrapper for OpenAI models (e.g. GPT-4o)      
from langchain_core.prompts import PromptTemplate   # The standard template class used to generate dynamic text prompts by inserting variables into a template
from langchain_core.prompts import  load_prompt     # 'load_prompt' loads JSON templates

load_dotenv('../keys/.env', override=True)          # Load variables from .env file into system environment
groq_api_key = os.getenv('GROQ_API_KEY')            # Retrieve Groq API key (used to access LLaMA model)
llama_model = ChatOpenAI(model="llama-3.3-70b-versatile", api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")



# Create a PromptTemplate and SAVE it to a JSON file
greet_template = PromptTemplate.from_template('Greet this person in five languages, one of which should be Urdu. The name of the person is {user_name}') 
greet_template.save("../data/greet_template.json") 

# Display the contents of the saved JSON file for debugging purpose
print("\nContents of saved ../data/greet_template.json File:\n")
with open("../data/greet_template.json", "r") as f:
    print(f.read())      

# Load the saved template back from the file
loaded_template = load_prompt("../data/greet_template.json")
your_name = input("Enter the name of the person you want to greet: ")
prompt = loaded_template.invoke({"user_name": your_name})

# At this point, 'prompt' contains the full prompt text ready to be sent to the LLM
response = llama_model.invoke(prompt)
print(response.content)


Contents of saved ../data/greet_template.json File:

{
    "name": null,
    "input_variables": [
        "user_name"
    ],
    "optional_variables": [],
    "output_parser": null,
    "partial_variables": {},
    "metadata": null,
    "tags": null,
    "template": "Greet this person in five languages, one of which should be Urdu. The name of the person is {user_name}",
    "template_format": "f-string",
    "validate_template": false,
    "_type": "prompt"
}


Enter the name of the person you want to greet:  Arif Butt


Here are greetings in five languages for Arif Butt:

1. **English**: Hello Arif Butt, how are you?
2. **Spanish**: Hola Arif Butt, ¿cómo estás?
3. **French**: Bonjour Arif Butt, comment allez-vous?
4. **Urdu**: السلام علیکم عارف بٹ، آپ کیسے ہو؟ (Assalam o Alaikum Arif Butt, aap kaise ho?)
5. **German**: Hallo Arif Butt, wie geht es Ihnen?

I hope this helps, and I'm sure Arif Butt will appreciate the diverse greetings.


## Example 6: A Research Paper Summarizer that uses LangChain `PromptTemplate` with Gradio

```
User selects options in UI
        ↓
Gradio passes inputs to summarize_paper()
        ↓
PromptTemplate formats structured prompt
        ↓
LLM (Llama 3.3 via Groq) generates summary
        ↓
Output returned to Gradio textbox
```

In [9]:
import os                                           # Access environment variables (e.g. os.getenv("API_KEY"))
from dotenv import load_dotenv                      # Load API keys from a .env file into environment variables         
from langchain_openai import ChatOpenAI             # LangChain wrapper for OpenAI models (e.g. GPT-4o)      
from langchain_core.prompts import PromptTemplate   # The standard template class used to generate dynamic text prompts by inserting variables into a template
import gradio as gr

load_dotenv('../keys/.env', override=True)          # Load variables from .env file into system environment
groq_api_key = os.getenv('GROQ_API_KEY')            # Retrieve Groq API key (used to access LLaMA model)
llama_model = ChatOpenAI(model="llama-3.3-70b-versatile", api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")

# ---------------------------------------------------------
# Define a PromptTemplate  having three user inputs:
   # `paper_input` → Selected research paper
   # `style_input` → Explanation style
   # `length_input` → Desired summary length
# ---------------------------------------------------------
template = PromptTemplate.from_template(
                                """
                                You are an AI research assistant.
                                Summarize the paper titled "{paper_input}" in a {style_input} style.
                                The summary should be {length_input}.
                                Make sure the explanation is clear and relevant.
                                """
                            )

# ---------------------------------------------------------
# Function to generate summary
# ---------------------------------------------------------
def summarize_paper(paper_input, style_input, length_input):
# Step 1: Create the prompt using the template
    prompt = template.invoke({
                                "paper_input": paper_input,
                                "style_input": style_input,
                                "length_input": length_input
                            })
    prompt_text = prompt.to_string()    # converts the prompt to a string object having the values passed by Gradio UI (for debugging)
# Step 2: Send the prompt to the LLM
    result = llama_model.invoke(prompt)
# Step 3: Return the prompt and the generated text
    return prompt_text, result.content


# ---------------------------------------------------------
# Build Gradio interface
# ---------------------------------------------------------
with gr.Blocks(title="Research Tool") as demo:
    gr.Markdown("# 🧠 Research Paper Summarizer")
    gr.Markdown("Select a research paper and customize the style and length of explanation.")

# Create a row layout: left column for inputs, right column for outputs
    with gr.Row():
        # -----------------------
        # Left Column: Inputs
        # -----------------------
        with gr.Column():
            # Dropdown to select the research paper
            paper_input = gr.Dropdown(
                choices=[
                    "Attention Is All You Need",
                    "BERT: Pre-training of Deep Bidirectional Transformers",
                    "GPT-3: Language Models are Few-Shot Learners",
                    "Diffusion Models Beat GANs on Image Synthesis"
                ],
                label="Select Research Paper",
                value="Attention Is All You Need"
            )
            # Dropdown to select explanation style
            style_input = gr.Dropdown(
                choices=[
                    "Beginner-Friendly",
                    "Technical",
                    "Code-Oriented",
                    "Mathematical"
                ],
                label="Explanation Style",
                value="Beginner-Friendly"
            )
            # Dropdown to select length of summary
            length_input = gr.Dropdown(
                choices=[
                    "Short (1-2 paragraphs)",
                    "Medium (3-5 paragraphs)",
                    "Long (detailed explanation)"
                ],
                label="Explanation Length",
                value="Medium (3-5 paragraphs)"
            )
            # Button to trigger the summarization
            summarize_btn = gr.Button("Summarize", variant="primary")
         # -----------------------
        # Right Column: Outputs
        # -----------------------
        with gr.Column():
            # Textbox to display the actual prompt sent to the model. Helps students see how the inputs are converted into a prompt
            prompt_output = gr.Textbox(
                label="Prompt Sent to Model",
                lines=4,
                placeholder="The actual prompt will appear here..."
            )
            # Textbox to display the LLM-generated summary
            summary_output = gr.Textbox(
                label="Summary",
                lines=15,
                placeholder="Click 'Summarize' to generate the explanation..."
            )

    # ------------------------------------------------------------------------------
    # When the 'Summarize' button is clicked:
    #        - pass the values from paper_input, style_input, length_input
    #        - call the 'summarize_paper' function
    #        - map the returned outputs to prompt_output and summary_output textboxes
    # --------------------------------------------------------------------------------
    summarize_btn.click(
                        fn=summarize_paper,
                        inputs=[paper_input, style_input, length_input],
                        outputs=[prompt_output, summary_output]  # map outputs to two textboxes
                        )

demo.launch(share=False, inbrowser=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


# <span style='background :lightgreen' >3. Role-Based Conversational Prompts with LangChain's `ChatPromptTemplate`</span>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">A ChatPromptTemplate in LangChain is conversations with system/user/assistant roles for multi-turn interactions, rather than just one plain text prompt.</h3>

- **Best For:**
    - Chatbots and virtual assistants
    - Customer support systems
    - Interactive AI applications
    - Role-playing scenarios with specific personas

```
| Message Type   | Purpose                                |  Typical Role                                                 |
| -------------- | -------------------------------------- | ------------------------------------------------------------- |
| SystemMessage  | Sets the context or behavior of the AI | e.g., “You are a helpful tutor who explains in simple terms.” |
| HumanMessage   | Captures user input                    | e.g., “What is LangChain?”                                    |
| AIMessage      | Represents the model’s response        | e.g., “LangChain is a framework for building LLM apps…”       |

```

## Example 7: Understanding LangChain messages: `SystemMessage`, `HumanMessage` and `AIMessage`

In [10]:
import os                                           # Access environment variables (e.g. os.getenv("API_KEY"))
from dotenv import load_dotenv                      # Load API keys from a .env file into environment variables         
from langchain_openai import ChatOpenAI             # LangChain wrapper for OpenAI models (e.g. GPT-4o)      
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage # Classes to represent chat roles

load_dotenv('../keys/.env', override=True)          # Load variables from .env file into system environment
groq_api_key = os.getenv('GROQ_API_KEY')            # Retrieve Groq API key (used to access LLaMA model)
llama_model = ChatOpenAI(model="llama-3.3-70b-versatile", api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")

# Create a Conversation
messages = [
            SystemMessage(content='You are a helpful assistant'),        # System instructions for the AI
            HumanMessage(content='Tell me about LangChain in one line')  # The user's question/input
            ]


result = llama_model.invoke(messages)              # Send all messages to the LLM and get a response
messages.append(AIMessage(content=result.content)) # Store AI's response as an AIMessage object


# # Iterate through all messages to display them with role-based formatting
print("\n================ Conversation Log ================\n")
for msg in messages:
    if isinstance(msg, SystemMessage):                  # Check if the message is a system message
        print(f"🧠 [System Message]: {msg.content}\n")
    elif isinstance(msg, HumanMessage):                 # Check if the message is from the user
        print(f"👤 [Human Message]: {msg.content}\n")
    elif isinstance(msg, AIMessage):                    # Check if the message is from the AI
        print(f"🤖 [AI Message]: {msg.content}\n")


================ Conversation Log ================

🧠 [System Message]: You are a helpful assistant

👤 [Human Message]: Tell me about LangChain in one line

🤖 [AI Message]: LangChain is an open-source framework that enables developers to build applications powered by large language models, providing a set of tools and APIs to interact with these models in a scalable and efficient manner.



## Example 8: Understanding LangChain `ChatPromptTemplate`

In [14]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate([
    ("system", "You are a helpful assistant"),
    ("user", "Tell me a joke about {topic}")
])

prompt_template.invoke({"topic": "dogs"})

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me a joke about dogs', additional_kwargs={}, response_metadata={})])

In [15]:
import os                                             # Access environment variables (e.g. os.getenv("API_KEY"))
from dotenv import load_dotenv                        # Load API keys from a .env file into environment variables         
from langchain_openai import ChatOpenAI               # LangChain wrapper for OpenAI models (e.g. GPT-4o)     
from langchain_core.prompts import ChatPromptTemplate # Used to create reusable structured prompts
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage # Classes to represent chat roles

load_dotenv('../keys/.env', override=True)          # Load variables from .env file into system environment
groq_api_key = os.getenv('GROQ_API_KEY')            # Retrieve Groq API key (used to access LLaMA model)
llama_model = ChatOpenAI(model="llama-3.3-70b-versatile", api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")


# Define a ChatPromptTemplate that allows creation of reusable prompts with variables {topic} is a placeholder that will be filled dynamically at runtime
prompt_template = ChatPromptTemplate.from_messages([
                                        ("system", "You are a helpful assistant."),  # System message defines assistant behavior
                                        ("user", "Tell me a joke about {topic}")     # User message contains variable placeholder
                                        ])

# Ask the user to enter a topic for the joke
mytopic = input("Enter the topic for joke: ")
# The invoke() method replaces template variables with actual values
myprompt = prompt_template.invoke({"topic": mytopic})

# Send the Prompt to the LLM  The invoke() method sends the formatted prompt to the LLM
response = llama_model.invoke(myprompt)
print(response.content)

Enter the topic for joke:  dogs


Why did the dog go to the vet?

Because he was feeling ruff.
